In [ ]:
import pandas as pd
import numpy as np

import requests
import os
from dotenv import load_dotenv

import concurrent.futures
import threading
import time

In [ ]:
#########################################
# STEP 1: CLEAN CSV TO UPLOAD NEW DATA  #
#########################################


df2 = pd.read_csv('unique_songs.csv')
df2['spotify track id'] = df2['track spotify url'].str.split(':', n=2).str[-1]

# create new columns for api call variables
added_columns = [
    "api track id", "id", "key", "mode", "camelot", "tempo", "duration",
    "popularity", "energy", "danceability", "happiness",
    "acousticness", "instrumentalness", "liveness",
    "speechiness", "loudness"
]

for col in added_columns:
    df2[col] = None

df2

In [ ]:
#########################################
# STEP 2: SET UP FOR GET API FUNCTION   #
#########################################
load_dotenv()
api_key = os.getenv("RAPIDAPI_KEY")

# initialize lock
lock = threading.Lock()
        
# set up api call session
headers = {
        'x-rapidapi-key': api_key,
        'x-rapidapi-host': 'track-analysis.p.rapidapi.com'
    }
session = requests.Session()
session.headers.update(headers)

# keep track of failed track requests
failed_tracks = []

In [ ]:
#########################################
# STEP 3: DEFINE GET API FUNCTION       #
#########################################

# 0 = under limit, 1 = limit hit
api_limit = 0

def get_api_data(i):
    # set spotifyId based on index i
    spotifyId = df2.at[i,'spotify track id']
    
    # configure api request
    url = f"https://track-analysis.p.rapidapi.com/pktx/spotify/{spotifyId}"
    
    try:
        # make api call
        result = session.get(url, timeout=20)
        
        # error check
        if result.status_code == 200:   # okay call
            output = result.json()
        else:
            output = {}
            print(f"Error {result.status_code}: \n{result.text}")
            with lock:
                failed_tracks.append(i)
    except requests.exceptions.RequestException as ex:
        output = {}
        print(f"API Call failed for track {i}: {ex}")
        with lock:
            failed_tracks.append(i)
    
    
    # thread safety to avoid corrupt data
    with lock:
        df2.at[i,'api track id'] = spotifyId
        for col in added_columns:
            if col == "api track id":
                df2.at[i,col] = spotifyId
            else:
                df2.at[i,col] = output.get(col)
    
    # buffer for rate limit
    time.sleep(3)
            
    # debugging setup
    return i

In [ ]:
#########################################
# STEP 4a: SAVE HEADER IN CSV FILE      #
#########################################

# df2.iloc[0:0].to_csv('progress_backup.csv', mode='w', header=True, index=False )

In [ ]:
#########################################
# STEP 4b: ITERATE API CALLS IN BATCHES #
#########################################

# batch variables
batch_start = 2900
batch_size = 50
batch_end = batch_start + batch_size
total_tracks = 27434
run_again = True

# signal start of api calls
print("Starting to Process Tracks.")
# while loop to continue running batches
while run_again == True:

    # api call & input through batch
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(get_api_data, i) for i in range(batch_start,batch_end)]
        
        for count, task in enumerate(concurrent.futures.as_completed(futures)):
            task.result()
        
    # save this batch's rows into csv file
    df2.iloc[batch_start:batch_end].to_csv('progress_backup.csv', mode='a', header=False, index=False )

    # check range of batch & assign batch_end value
    if batch_end == total_tracks:
        run_again = False
    elif (batch_end + batch_size) > total_tracks:
        print(f"Songs {batch_start} to {batch_end} processed!")
        batch_start += batch_size
        batch_end = total_tracks
    else:
        print(f"Songs {batch_start} to {batch_end} processed!")
        batch_start = batch_start + batch_size
        batch_end = batch_start + batch_size

# signal end of api calls
print("Finished processing all tracks!")
print(f"{len(failed_tracks)} failed.")
print(failed_tracks)

# done: 2900
